# 🌱 Evidence Lab: the susceptible variety
### Break the analysis, repair it, and defend one scientific claim

**One real field experiment · 12 plots · 4 blocks · 3 treatments · 90 minutes**

<div style="padding:12px;border-radius:10px;background:#eef7f2;text-align:center;font-size:17px">
👀 IMAGE → 🧩 UNIT → 🎲 RANDOMIZE → 📊 COMPARE → 🔧 BREAK → 🗺️ CHECK → ✍️ DEFEND
</div>

All twelve plots from one **susceptible sugar-beet variety** are included
(*susceptible* = easily affected by disease).
It was chosen because of low disease resistance—not because its results looked significant.

**Core route:** about 75 minutes · **Optional CNN:** 10 minutes · **Exit:** 5 minutes  
**45-minute express route:** run only sections marked ⭐.

## How this notebook works

**Predict → change something → run → inspect → decide.**

Work in pairs. The **Agronomist** explains the field; the **Analyst** changes a dropdown or slider and runs the cell. Swap roles halfway.

Nothing below is pre-executed. Your predictions and choices stay in this Colab session.

In [ ]:
#@title 0 · Setup — run once { display-mode: "form" }
from pathlib import Path
from urllib.request import urlopen
import importlib, sys

BASE = "https://raw.githubusercontent.com/Facuriy/agro-data-analysis-hands-on/v2.2.4"
if not Path("agro_helpers.py").exists() and not Path("data").exists():
    Path("agro_helpers.py").write_bytes(urlopen(f"{BASE}/agro_helpers.py", timeout=25).read())
import agro_helpers
importlib.reload(agro_helpers)
lab = agro_helpers.EvidenceLab(BASE)
lab.versions()
lab.opening_field()

## 👀 First look · What can this image tell us? ⏱ 3 min

Use only the RGB orthomosaic shown by Setup. Do not diagnose a
treatment yet: first separate **what you see** from **what you infer**.

In [ ]:
#@title Record one observation and one limit { display-mode: "form" }
visible_pattern = "Choose..." #@param ["Choose...", "mostly uniform", "patchy: some areas are greener than others"]
image_proves_treatment = "Choose..." #@param ["Choose...", "yes", "no", "not sure"]
first_observation = "I see..." #@param {type:"string"}
print("OBSERVATION:", first_observation)
print("PATTERN:", visible_pattern)
if image_proves_treatment == "no":
    print("✅ Correct limit: RGB can suggest a hypothesis; treatment records identify treatments.")
else:
    print("LIMIT: RGB can suggest a hypothesis; treatment records are needed to identify treatments.")

## ⭐ Mission 1 · Treatment detective ⏱ 8 min

First learn what the labels mean:

| Treatment | What happened in this teaching subset? |
|---|---|
| **Control** | No deliberate inoculation and no recorded fungicide program. |
| **Fungicide** | Received the recorded fungicide program; not deliberately inoculated. |
| **Inoculated** | Deliberately inoculated to increase disease pressure; no fungicide treatment. |

Now zoom into three September RGB crops from the **same block**.
Use each treatment label **once**.
Which is which? Visual symptoms can guide a hypothesis, but they
cannot prove a treatment effect.

In [ ]:
lab.mystery_plots()

In [ ]:
#@title Save your treatment guesses { display-mode: "form" }
mystery_A = "Choose..." #@param ["Choose...", "Control", "Fungicide", "Inoculated"]
mystery_B = "Choose..." #@param ["Choose...", "Control", "Fungicide", "Inoculated"]
mystery_C = "Choose..." #@param ["Choose...", "Control", "Fungicide", "Inoculated"]
treatment_guesses = {"A": mystery_A, "B": mystery_B, "C": mystery_C}
if "Choose..." in treatment_guesses.values():
    print("⚠️ Choose one treatment for A, B and C before the reveal.")
elif len(set(treatment_guesses.values())) < 3:
    print("⚠️ Use Control, Fungicide and Inoculated once each.")
else:
    print("✅ Hypotheses saved. Now run the reveal.")

In [ ]:
lab.reveal_treatments(treatment_guesses)

## ⭐ RGB lab · Image → number → green-area proxy ⏱ 12 min

Before calculating anything, inspect the experiment inventory:

- **12 field plots** = 4 blocks × 3 treatment types;
- **2 RGB dates** = 26 June (81 DAS) and 3 September (150 DAS);
- each block contains one plot of each treatment.

In [ ]:
lab.experiment_overview()

### 1 · Choose an RGB vegetation index

| Index | Simple idea |
|---|---|
| **ExG** | how much green exceeds red + blue |
| **GLI** | normalized green balance using all RGB channels |
| **NGRDI** | normalized green–red contrast |

These are RGB indices—not NDVI. Select a palette and normalization
to change how the map looks. Then classify green pixels using either
zero or an automatic Otsu threshold.

**Normalization:** fixed range = easier comparison · crop min–max =
strongest local contrast · robust 2–98% = reduces the influence of
extreme pixels. Palette and normalization affect visualization;
index and threshold define the green-pixel calculation.

In [ ]:
#@title Build one vegetation-index map { display-mode: "form" }
plot_for_index = "P03" #@param ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11", "P12"]
rgb_date = "September" #@param ["June", "September"]
vegetation_index = "ExG" #@param ["ExG", "GLI", "NGRDI"]
color_palette = "RdYlGn" #@param ["RdYlGn", "Greens", "viridis", "gray"]
color_normalization = "robust 2–98%" #@param ["robust 2–98%", "fixed index range", "crop min–max"]
green_pixel_rule = "index > 0" #@param ["index > 0", "Otsu per crop"]
index_result = lab.vegetation_index_lab(
    plot_for_index, rgb_date, vegetation_index,
    color_palette, color_normalization, green_pixel_rule
)

**Two-run challenge:** keep the plot, date, index and threshold fixed;
change only palette or normalization and rerun. The colors change,
but the green-pixel percentage should not. Then change the index or
threshold: now the number may change.

### 2 · Estimate green-area change

Use the **same index and one shared threshold** for both dates.
Compare green preview pixels and green share—not physical square
metres. Each pair should choose a different plot. Which plots lose
green and which gain it?

In [ ]:
#@title Compare June with September { display-mode: "form" }
plot_for_change = "P03" #@param ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11", "P12"]
change_index = "ExG" #@param ["ExG", "GLI", "NGRDI"]
shared_threshold = "shared Otsu threshold" #@param ["shared Otsu threshold", "index > 0"]
green_change = lab.green_area_change(plot_for_change, change_index, shared_threshold)

**Scientific limit:** the images are downsampled JPEG previews and
were stretched separately. This exercise teaches segmentation and
change calculation; it does **not** estimate calibrated canopy loss.

### 3 · Connect image descriptors to agronomic variables

In [ ]:
#@title Identify today’s outcome, then reveal the maps { display-mode: "form" }
outcome_guess = "Choose..." #@param ["Choose...", "September green share", "disease severity", "sugar content"]
lab.evidence_maps()
print("✅ Today’s outcome is sugar content." if outcome_guess == "sugar content"
      else "Look again: green share describes the image; severity is an ordered post-treatment score; sugar content is the outcome.")

**Three roles:** green share is a continuous image descriptor; severity is a post-treatment ordered score; sugar content is today’s continuous outcome. Severity must not be used to “adjust away” the treatment response.

## ⭐ Mission 2 · Copy-paste trap (pseudoreplication) ⏱ 8 min

The treatment was assigned to a **field plot**. Copying pixels or rows does not create new experimental units.

Run once with **1 copy**, then move the slider to **50** and run again.
What happens to the confidence interval when identical rows are falsely counted as new evidence?

*The dot is the estimated difference; the horizontal line is its 95% confidence interval. A wider line means more uncertainty.*

In [ ]:
#@title Copy every row and pretend each copy is independent { display-mode: "form" }
fake_copies = 1 #@param {type:"slider", min:1, max:100, step:1}
lab.pseudoreplication(fake_copies)

**Decision:** the effect estimate barely moves, but the fake interval collapses. More rows are not automatically more independent evidence.

## ⭐ Mission 3 · The baseline trap ⏱ 7 min

Run the timeline, then decide whether June green share is a clean
pre-treatment baseline.

In [ ]:
lab.timeline()

In [ ]:
#@title Decide from the timeline { display-mode: "form" }
june_baseline_decision = "Choose..." #@param ["Choose...", "yes", "no", "not sure"]
if june_baseline_decision == "no":
    print("✅ Correct: treatment had already begun, and within-day order is unknown.")
elif june_baseline_decision == "Choose...":
    print("⚠️ Read the event order above and choose an answer.")
else:
    print("Look again: inoculation occurred before the June image, and Fungicide 1 shares its date.")

June RGB was two days after the recorded inoculation timing and on the same day as the first fungicide application; within-day order is unknown.

**Decision:** use `treatment + block` as the primary analysis. Adding June green answers a different conditional question and belongs only in sensitivity analysis.

## ⭐ Mission 4 · Randomize, then model ⏱ 18 min

Before any test, use the evidence maps from Mission 1 to predict
the **direction** of each sugar-content difference.

In [ ]:
#@title Predict directions before seeing the tests { display-mode: "form" }
fungicide_vs_control = "Choose..." #@param ["Choose...", "higher", "equal", "lower"]
inoculated_vs_control = "Choose..." #@param ["Choose...", "higher", "equal", "lower"]
if "Choose..." in (fungicide_vs_control, inoculated_vs_control):
    print("⚠️ Choose both directions before continuing.")
else:
    print("✅ Predictions saved. Now test them.")

Each block contains one plot of each treatment. For the
**Inoculated − Control** comparison, there are 6 allowed label orders
per block and `6⁴ = 1,296` reshuffles in total.

In [ ]:
lab.randomization_test()

**Read the randomization p-value:** it is the proportion of legal
label assignments producing a difference at least this extreme
under the no-treatment-effect randomization model. It is **not** the
probability that the null hypothesis is true, and it is not the
effect size.

### Choose the analysis

| What we have | What we use |
|---|---|
| continuous sugar content + 3 assigned treatments + 4 blocks | blocked linear model |
| 3 pairwise treatment differences | Holm correction for the 3 p-values |

**Read the plot:** dot = estimated difference · line = 95% interval ·
zero = no difference · **pp** = percentage points.

In [ ]:
contrasts = lab.analyze_field(fungicide_vs_control, inoculated_vs_control)

Read every comparison in this order: **direction → size → interval → Holm p-value**. Holm is used because we examined all three treatment pairs. Sugar **content** is not yet sugar yield or profit.

**Compare methods:** the randomization and linear-model results use
different statistics and assumptions. With four blocks, the exact
test has only a few possible p-values. Report the estimate and
uncertainty, not only “significant/not significant”, and never
choose whichever p-value is smaller after seeing both.

The 95% intervals shown are ordinary model intervals; Holm adjusts
the three p-values, not those intervals. Also, Fungicide − Inoculated
compares two treatment packages—it is not a pure “fungicide effect”.

<details><summary>Why is block kept even if its p-value is large?</summary>
Block belongs to the randomized design. It is not selected or removed by significance.
</details>

### Optional sensitivity · ANCOVA

June green is **not** a clean baseline, so ANCOVA is not the primary
model here. Run it only to ask: *does conditioning on this imperfect
early image change the treatment estimates?* Do not call this a
baseline-adjusted causal effect. ANCOVA also assumes an approximately
linear relationship with a common slope across treatments.

In [ ]:
#@title Predict, then run ANCOVA sensitivity { display-mode: "form" }
ancova_reverses_directions = "Choose..." #@param ["Choose...", "yes", "no", "not sure"]
lab.ancova_sensitivity()
print("✅ The directions do not reverse." if ancova_reverses_directions == "no"
      else "Decision: directions do not reverse; ANCOVA still does not repair the timing problem.")

## Mission 5 · Try to break the conclusion ⏱ 16 min

### A. Remove one plot

Choose any plot. Compare **direction, magnitude and interval width**
with the all-plot result. More than one can change. This is a
sensitivity check—not a license to delete inconvenient observations.

In [ ]:
#@title Choose one plot to remove { display-mode: "form" }
plot_to_remove = "P01" #@param ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11", "P12"]
lab.drop_one_plot(plot_to_remove)

### B. Correlation trap

Before revealing the two correlations: will the strong overall
green–sugar association remain strong **inside** each treatment?

In [ ]:
#@title Predict, then reveal { display-mode: "form" }
within_correlation_prediction = "Choose..." #@param ["Choose...", "about the same", "much weaker", "stronger"]
lab.correlation_trap()
print("✅ Yes: it is much weaker inside treatments." if within_correlation_prediction == "much weaker"
      else "Decision: it becomes much weaker inside treatments; group separation creates much of the overall pattern.")
print("Neither correlation proves that green share causes sugar content.")

### C. Health and space

**Model assumptions, in plain language:** plots do not interfere;
residual spread and shape are reasonably stable; block and treatment
effects add. With one plot per treatment × block, their interaction
cannot be estimated separately.

Read left to right: random residual cloud · points near the Q–Q line ·
Cook values above the guide deserve inspection. Then look for
same-sign residuals clustering in one field area. Independence comes
from design—not from a residual test. With only 12 plots, diagnostics
are clues, not certificates; Cook > 4/n is not a deletion rule.

In [ ]:
influential_plots = lab.model_health()

In [ ]:
lab.residual_map()

In [ ]:
#@title Record your diagnostic decision { display-mode: "form" }
model_health_decision = "Choose..." #@param ["Choose...", "🟢 no obvious problem", "🟡 caution / inspect", "🔴 rethink the analysis"]
spatial_cluster = "Choose..." #@param ["Choose...", "yes", "no", "not sure"]
print("A defensible choice is 🟡 caution: inspect influential plots, but do not delete them automatically.")

The residual map uses preview-pixel coordinates. It is a qualitative spatial check, not a spatial model and not proof of independence.

## ⭐ Mission 6 · Defend the claim ⏱ 8 min

Sort each card:

| Card | RESULT · DISCUSSION · NOT SUPPORTED? |
|---|---|
| Inoculated plots had lower estimated sugar content than Control. | |
| The pathogen directly caused the entire sugar difference. | |
| The susceptible variety and single field season limit generalization. | |
| An independent field and year should be tested next. | |

<details><summary>Open only after your pair has decided</summary>
<b>RESULT:</b> first · <b>NOT SUPPORTED:</b> second · <b>DISCUSSION:</b> third and fourth.
</details>

Random assignment supports a causal claim about the **assigned
treatment package in this experiment**, assuming no interference.
It does not prove a biological mechanism or generalize to every field.

In [ ]:
#@title Save your scientific conclusion { display-mode: "form" }
your_result = "Compared with Control, ... differed by ... pp (95% CI ... to ...)." #@param {type:"string"}
your_discussion = "This may indicate ..., but this one variety and season limit ...; next test ...." #@param {type:"string"}
print("RESULT:", your_result)
print("DISCUSSION:", your_discussion)

### Peer-review exchange

Swap conclusions with another pair. Award one point for each:

- [ ] includes an estimate or 95% interval;
- [ ] avoids an unmeasured causal mechanism;
- [ ] names one concrete limitation and one next test.

## Optional Mission · CNN evaluation ⏱ 10 min / homework

CNN theory belongs to the preceding course. Here we ask whether the evaluation is honest. If class time is short, skip directly to the exit check; CNN is not assessed there.

**Read the evidence:** 1-in-3 accuracy is a descriptive chance reference · rows
are true classes · columns are CNN predictions · testing a truly new
field is stronger than reusing this field.

In [ ]:
#@title Choose a held-out block { display-mode: "form" }
held_out_block = 1 #@param {type:"slider", min:1, max:4, step:1}
lab.inspect_cnn_fold(held_out_block)
print("Test set: 3 whole plots × 2 dates = 6 images. None of those plots may enter training.")

In [ ]:
#@title Predict which split will score higher { display-mode: "form" }
higher_toy_score = "Choose..." #@param ["Choose...", "random image rows", "whole field blocks", "same score"]
more_honest_split = "Choose..." #@param ["Choose...", "random image rows", "whole field blocks", "either"]
lab.leakage_demo()
if higher_toy_score == "random image rows" and more_honest_split == "whole field blocks":
    print("✅ Correct: random rows score higher here, but whole blocks are more honest.")
else:
    print("Decision: random rows can score higher by memorizing plot fingerprints; whole blocks are more honest.")

In [ ]:
lab.evaluate_cnn()

The leakage demo is a deliberately cheating **plot-ID lookup model**, not the CNN. A random row split rewards memorization when the other date from the same plot enters training. Holding out a whole block removes those plot fingerprints. A CNN can learn analogous background or spatial signatures even without receiving the plot ID explicitly.

The CNN table contains grouped-fold predictions. Compare class-specific
errors, accuracy and macro-F1—not only total correct. The 24 rows are
12 plots observed twice, not 24 independent field units. No binomial
p-value is used because these predictions are not independent trials.
This is still not independent deployment validation.

In [ ]:
#@title Deployment decision { display-mode: "form" }
deploy_cnn = "Choose..." #@param ["Choose...", "deploy now", "do not deploy yet"]
deployment_reason = "Because..." #@param {type:"string"}
print("Decision saved:", deploy_cnn, deployment_reason)
print("A truly new field is still needed before deployment.")

## Bonus sandboxes · design the next experiment

These are optional self-study activities.

- **Producer view:** content is not yield, and yield is not profit.
- **Power sandbox:** power is the chance of detecting a chosen effect
  under the planning assumptions. Change blocks and the target
  difference in percentage points; treat this as planning, not a guarantee.

In [ ]:
lab.producer_view()

In [ ]:
#@title Design the next trial { display-mode: "form" }
future_blocks = 8 #@param {type:"slider", min:3, max:12, step:1}
target_difference_pp = 0.30 #@param {type:"slider", min:0.10, max:1.00, step:0.05}
lab.power_sandbox(future_blocks, target_difference_pp)

## Exit check + evidence ticket ⏱ 5 min

Every question below was taught in the core route. Choose an answer;
no correct option is preselected.

1. What was the experimental unit?
2. Was June a clean before-treatment measurement?
3. What does a p-value describe?
4. Does a strong overall correlation prove the same relationship inside treatments?
5. Which claim is supported?

In [ ]:
#@title Check your decisions { display-mode: "form" }
exit_unit = "Choose..." #@param ["Choose...", "pixel", "image crop", "field plot"]
exit_baseline = "Choose..." #@param ["Choose...", "yes", "no", "not sure"]
exit_pvalue = "Choose..." #@param ["Choose...", "probability the null is true", "data compatibility under the null model", "size of the effect"]
exit_correlation = "Choose..." #@param ["Choose...", "yes", "no"]
exit_claim = "Choose..." #@param ["Choose...", "treatment differences in this experiment", "the same effect in every future field", "the pathogen mechanism is proven"]
exit_answers = {"unit": exit_unit, "baseline": exit_baseline,
                "pvalue": exit_pvalue, "correlation": exit_correlation,
                "claim": exit_claim}
lab.score_quiz(exit_answers)

### Evidence card to take away

For any new experiment, ask:

1. What is the experimental unit?
2. How were treatments randomized or grouped?
3. What is the outcome—and what is only a descriptor?
4. Was each covariate measured before treatment?
5. What claim is supported, and what remains interpretation?

**Final challenge:** name one conclusion this dataset permits and one it cannot support.

<details><summary>Example after you answer</summary>
Permitted: treatment differences in sugar content were estimated for
this susceptible variety in this experiment. Not permitted: the same
effect will occur in every variety, field and year.
</details>